# Table-reading VLM agent: what helps, what doesn't

A DeepSeek "brain" decides when to call two tools: a Qwen2.5-VL-3B table reader and a deterministic Python solver.
Task: *remove one song, re-rank the rest per judge, sum the scores*, on synthetic score-table images
(clean and degraded). See `README.md` in this folder for results and conclusions.

**Setup**
- Colab runtime: **T4 GPU**
- Add a Colab secret named `DEEPSEEK_API_KEY` (key icon in the left bar) and enable notebook access. Never paste the key into the notebook.
- Approximate API cost for the whole notebook: roughly 0.08 USD (see README for per-experiment costs). Qwen runs locally on the GPU.

In [ ]:
!pip install -q openai transformers accelerate qwen-vl-utils pillow bitsandbytes matplotlib

## 1. Models: Qwen2.5-VL-3B (eyes, local, 4-bit) and DeepSeek (brain, API)

In [ ]:
import os, re, json, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from IPython.display import display
from google.colab import userdata
from openai import OpenAI
from transformers import BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- brain: DeepSeek via the OpenAI-compatible API ---
client = OpenAI(api_key=userdata.get("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")
LLM_MODEL = "deepseek-flash"
PRICE_IN, PRICE_OUT = 0.15, 0.6      # USD per million tokens (flash, off-peak; check the pricing page)

# --- eyes: Qwen2.5-VL-3B, 4-bit NF4 (about 2.4 GB on a T4) ---
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
processor = AutoProcessor.from_pretrained(model_id)
print("device:", model.device, f"| GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

def ask(image_path, question, max_new_tokens=128):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": f"file://{image_path}", "max_pixels": 640 * 640},
        {"type": "text", "text": question}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    inputs = processor(text=[text], images=images, videos=videos, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)   # greedy: Qwen is deterministic
    out = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(out, skip_special_tokens=True)[0]

## 2. Data: synthetic tables, ground truth by code, and image degradation

Each judge gives every song a distinct score. After removing one song, each judge re-ranks the rest
(best gets n points, worst gets 1). Ground truth is computed by `rerank_without`, so no hand labels.

`degrade(..., "mild")`: downscale, light blur, JPEG. `degrade(..., "hard")`: more downscaling, blur, small rotation, noise.

In [ ]:
os.makedirs("/content/eval", exist_ok=True)

def make_table(n_songs, n_judges, rng):
    names = [chr(65 + i) for i in range(n_songs)]
    table = {s: [] for s in names}
    for _ in range(n_judges):
        scores = list(range(1, n_songs + 1))
        rng.shuffle(scores)
        for s, v in zip(names, scores):
            table[s].append(v)
    return table

def draw_table(table, path):
    n_judges = len(next(iter(table.values())))
    cols = [f"J{j+1}" for j in range(n_judges)] + ["Tot."]
    cells = [[*v, sum(v)] for v in table.values()]
    fig, ax = plt.subplots(figsize=(1 + 0.8 * n_judges, 0.8 + 0.5 * len(table)))
    ax.axis("off")
    ax.table(cellText=cells, rowLabels=list(table), colLabels=cols, loc="center").scale(1, 1.8)
    plt.title("Song contest: judges' scores", pad=8)
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)

def rerank_without(table, removed):
    songs = [s for s in table if s != removed]
    n = len(songs)
    totals = {s: 0 for s in songs}
    for j in range(len(table[songs[0]])):
        order = sorted(songs, key=lambda s: table[s][j], reverse=True)
        for rank, s in enumerate(order):
            totals[s] += n - rank
    return totals

def make_bench(seed, n, prefix):
    rng = random.Random(seed); out = []
    for k in range(n):
        n_songs, n_judges = rng.choice([4, 5]), rng.choice([5, 6, 7])
        table = make_table(n_songs, n_judges, rng)
        path = f"/content/eval/{prefix}{k+1}.png"
        draw_table(table, path)
        removed = rng.choice(list(table))
        others = [s for s in table if s != removed]
        totals = rerank_without(table, removed)
        if k % 2 == 0:
            kind, target, answer = "winner", None, max(totals.values())
            q = (f"如果取消歌曲{removed}，剩下的歌曲每位评委重新按分数从高到低赋{len(others)}到1分"
                 f"（最高分得{len(others)}分），获胜歌曲的总分是多少？")
        else:
            kind, target = "target", rng.choice(others)
            answer = totals[target]
            q = (f"如果取消歌曲{removed}，剩下的歌曲每位评委重新按分数从高到低赋{len(others)}到1分"
                 f"（最高分得{len(others)}分），歌曲{target}的总分是多少？")
        out.append({"image": path, "question": q, "answer": answer, "true_table": table,
                    "removed": removed, "target": target, "kind": kind})
    return out

def degrade(src, dst, level):
    img = Image.open(src).convert("RGB")
    w, h = img.size
    if level == "mild":
        img = img.resize((int(w * 0.45), int(h * 0.45)), Image.BILINEAR)
        img = img.filter(ImageFilter.GaussianBlur(0.8))
        quality = 35
    else:  # hard
        img = img.resize((int(w * 0.40), int(h * 0.40)), Image.BILINEAR)
        img = img.filter(ImageFilter.GaussianBlur(0.9))
        img = img.rotate(2.5, expand=True, fillcolor=(255, 255, 255))
        arr = np.asarray(img).astype(np.float32)
        arr += np.random.default_rng(0).normal(0, 8, arr.shape)
        img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
        quality = 40
    img.save(dst, "JPEG", quality=quality)

def upscale(src, dst, k=2):
    img = Image.open(src).convert("RGB")
    img = img.resize((img.width * k, img.height * k), Image.LANCZOS)
    img.save(dst, "JPEG", quality=95)

# main benchmark: 8 tables (seed 42). Same tables as the first experiment in the repo root.
bench = make_bench(42, 8, "g")
for b in bench:
    print(b["question"][:40], "...", "answer =", b["answer"])
degrade(bench[0]["image"], "/content/eval/demo_mild.jpg", "mild")
degrade(bench[0]["image"], "/content/eval/demo_hard.jpg", "hard")
display(Image.open("/content/eval/demo_mild.jpg")); print("mild")
display(Image.open("/content/eval/demo_hard.jpg")); print("hard")

## 3. Tools and the agent loop

- `read_table`: Qwen converts the table image to JSON. Returns the table plus warnings (row lengths differ).
- `compute_totals`: deterministic solver; also returns the maximum possible total as a sanity bound.
- The loop (`run_agent`): the model chooses a tool, sees the result, and decides whether to continue or answer.

In [ ]:
def parse_final(text):
    m = re.findall(r"最终答案\D{0,4}?(-?\d+(?:\.\d+)?)", text or "")
    return float(m[-1]) if m else None

def is_ok(ans, truth):
    return ans is not None and abs(ans - truth) < 1e-6

def read_table(image_path, hint=""):
    prompt = ("把图中的表格转成JSON：键是行名，值是该行从左到右的数字列表。"
              "不要包含Tot.（总分）列。" + hint + "只输出JSON，不要其他文字。")
    raw = ask(image_path, prompt, 300)
    try:
        table = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
    except Exception as e:
        return {"error": f"parse failed: {e}", "raw": raw}
    lens = {k: len(v) for k, v in table.items()}
    warnings = []
    if len(set(lens.values())) > 1:
        warnings.append(f"rows have different lengths: {lens}")
    return {"table": table, "row_lengths": lens, "warnings": warnings}

def compute_totals(table, removed):
    if removed not in table:
        return {"error": f"song {removed} not in table {list(table)}"}
    totals = rerank_without(table, removed)
    n_judges = len(table[next(iter(table))])
    return {"totals": totals, "max_possible_per_song": len(totals) * n_judges}

TOOL_FUNCS_V1 = {"read_table": read_table, "compute_totals": compute_totals}
COMPUTE_SCHEMA = {"type": "function", "function": {
    "name": "compute_totals",
    "description": "取消一首歌，剩下的歌每位评委重新按分数从高到低赋分，返回每首歌总分和每首歌可能的最大总分。",
    "parameters": {"type": "object", "properties": {
        "table": {"type": "object", "additionalProperties": {"type": "array", "items": {"type": "number"}}},
        "removed": {"type": "string"}},
        "required": ["table", "removed"]}}}
TOOLS_V1 = [
    {"type": "function", "function": {
        "name": "read_table",
        "description": "用视觉模型读取图片里的评分表，返回 JSON 表格（行名 -> 各评委分数列表）和警告信息。读错时可以带 hint 重新读。",
        "parameters": {"type": "object", "properties": {
            "image_path": {"type": "string"},
            "hint": {"type": "string", "description": "可选，补充表格结构，如'共4行，每行5个分数'"}},
            "required": ["image_path"]}}},
    COMPUTE_SCHEMA,
]
SYSTEM_V1 = ("你是处理评分表图片的助手。先用 read_table 读图，检查返回的表格是否合理（行长度一致、分数在合理范围内），"
             "不合理就带 hint 重读；再用 compute_totals 计算，不要心算；如果结果超过 max_possible_per_song 或明显不合理，回头检查。"
             "最后一行以“最终答案：”开头，只写一个数字。")

def run_agent(question, image_path, tools, tool_funcs, system, max_steps=10):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": f"{question}\n图片路径：{image_path}"}]
    tin = tout = 0; steps = []; final = None
    for step in range(max_steps):
        r = client.chat.completions.create(model=LLM_MODEL, messages=messages, tools=tools)
        tin += r.usage.prompt_tokens; tout += r.usage.completion_tokens
        msg = r.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            final = msg.content; break
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            result = tool_funcs[call.function.name](**args)
            steps.append({"tool": call.function.name, "args": args, "result": result})
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result, ensure_ascii=False)})
    cost = tin / 1e6 * PRICE_IN + tout / 1e6 * PRICE_OUT
    return {"answer": parse_final(final), "final": final, "steps": steps,
            "tokens_in": tin, "tokens_out": tout, "cost": cost}

def solve(b, table):
    try:
        totals = rerank_without(table, b["removed"])
        return max(totals.values()) if b["kind"] == "winner" else totals[b["target"]]
    except Exception:
        return None

def run_fixed(b, image_path):
    """Fixed workflow: read the table once, then compute. No checking, no retry."""
    out = read_table(image_path)
    if "table" not in out:
        return None, out
    return solve(b, out["table"]), out

def reads_of(steps):
    reads = [s for s in steps if s["tool"] == "read_table"]
    last = next((s["result"]["table"] for s in reversed(reads) if "table" in s["result"]), None)
    return reads, last

## 4. Experiment 1: clean images (agent loop, 8 tables)

Reference: a pure VLM (no tools) scored 0/8 on these tables (see the notebook in the repo root).
Here the table is read by Qwen and the arithmetic is done by the solver; the agent decides the order of calls.

In [ ]:
exp1 = []
for i, b in enumerate(bench):
    fx_ans, fx_out = run_fixed(b, b["image"])
    res = run_agent(b["question"], b["image"], TOOLS_V1, TOOL_FUNCS_V1, SYSTEM_V1, max_steps=8)
    reads, last = reads_of(res["steps"])
    exp1.append({"i": i, "truth": b["answer"], "fixed": fx_ans, "fixed_ok": is_ok(fx_ans, b["answer"]),
                 "agent": res["answer"], "agent_ok": is_ok(res["answer"], b["answer"]),
                 "table_exact": last == b["true_table"], "n_reads": len(reads),
                 "tools": [s["tool"] for s in res["steps"]],
                 "tokens_in": res["tokens_in"], "tokens_out": res["tokens_out"], "cost": res["cost"]})
    r = exp1[-1]
    print(f"Q{i+1}: truth={r['truth']} fixed={r['fixed']} {'OK' if r['fixed_ok'] else 'X'} | "
          f"agent={r['agent']} {'OK' if r['agent_ok'] else 'X'} | table_exact={r['table_exact']} | tools={r['tools']} | ${r['cost']:.5f}")
print(f"\nfixed {sum(r['fixed_ok'] for r in exp1)}/8 | agent {sum(r['agent_ok'] for r in exp1)}/8 | "
      f"agent cost ~ ${sum(r['cost'] for r in exp1):.4f}")
json.dump(exp1, open("/content/exp1_clean.json", "w"), ensure_ascii=False, indent=2)

## 5. Experiment 2: degraded images, fixed workflow vs agent (same 8 tables, mild and hard)

The fixed workflow reads once and computes. The agent can check the result and re-read with a hint.
Qwen is deterministic, so both start from the same first read. The agent can only do better if a re-read changes the table.

In [ ]:
exp2 = []
for lvl in ["mild", "hard"]:
    for i, b in enumerate(bench):
        dst = f"/content/eval/g{i+1}_{lvl}.jpg"
        degrade(b["image"], dst, lvl)
        fx_ans, fx_out = run_fixed(b, dst)
        fx_exact = ("table" in fx_out) and fx_out["table"] == b["true_table"]
        res = run_agent(b["question"], dst, TOOLS_V1, TOOL_FUNCS_V1, SYSTEM_V1, max_steps=8)
        reads, last = reads_of(res["steps"])
        exp2.append({"level": lvl, "i": i, "truth": b["answer"],
                     "fixed": fx_ans, "fixed_ok": is_ok(fx_ans, b["answer"]), "fixed_table_exact": fx_exact,
                     "agent": res["answer"], "agent_ok": is_ok(res["answer"], b["answer"]),
                     "agent_table_exact": last == b["true_table"], "n_reads": len(reads),
                     "hints": [s["args"].get("hint", "") for s in reads],
                     "cost": res["cost"], "final": res["final"]})
        r = exp2[-1]
        print(f"[{lvl}] Q{i+1}: truth={r['truth']} | fixed={r['fixed']} {'OK' if r['fixed_ok'] else 'X'} (exact={fx_exact})"
              f" | agent={r['agent']} {'OK' if r['agent_ok'] else 'X'} (exact={r['agent_table_exact']}, reads={r['n_reads']})")
print()
for lvl in ["mild", "hard"]:
    rs = [r for r in exp2 if r["level"] == lvl]
    print(f"{lvl}: fixed {sum(r['fixed_ok'] for r in rs)}/8 (table exact {sum(r['fixed_table_exact'] for r in rs)}/8) | "
          f"agent {sum(r['agent_ok'] for r in rs)}/8 (table exact {sum(r['agent_table_exact'] for r in rs)}/8) | "
          f"agent re-reads on {sum(r['n_reads'] > 1 for r in rs)} questions")
print(f"agent cost ~ ${sum(r['cost'] for r in exp2):.4f}")
json.dump(exp2, open("/content/exp2_degraded.json", "w"), ensure_ascii=False, indent=2)

## 6. Experiment 3: a checksum for the agent (Tot. column)

Idea: also read the `Tot.` column. If a row's scores do not sum to its `Tot.`, the read is wrong, and the tool returns a warning.
Two variants share the same warning: a **scripted** workflow (code-written retry, at most 2 re-reads) and the **agent** (model decides).
This separates "is the checksum useful" from "is the agent useful". Note: the code assumes `Tot.` is the last number in each row.

In [ ]:
def read_table_v2(image_path, hint=""):
    prompt = ("把图中的表格转成JSON：键是行名，值是该行从左到右的所有数字列表，"
              "包括最右边 Tot. 列的总分（它是每行的最后一个数字）。" + hint + "只输出JSON，不要其他文字。")
    raw = ask(image_path, prompt, 400)
    try:
        full = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
        table = {k: v[:-1] for k, v in full.items()}
        totals = {k: v[-1] for k, v in full.items()}
    except Exception as e:
        return {"error": f"parse failed: {e}", "raw": raw}
    lens = {k: len(v) for k, v in table.items()}
    warnings = []
    if len(set(lens.values())) > 1:
        warnings.append(f"rows have different lengths: {lens}")
    try:
        bad = [k for k, v in table.items() if sum(v) != totals[k]]
    except Exception as e:
        bad = []
        warnings.append(f"checksum failed: {e}")
    if bad:
        warnings.append(f"checksum mismatch in rows {bad}: sum of scores != Tot. column")
    return {"table": table, "totals_read": totals, "row_lengths": lens, "bad_rows": bad, "warnings": warnings}

TOOL_FUNCS_V2 = {"read_table": read_table_v2, "compute_totals": compute_totals}
TOOLS_V2 = [
    {"type": "function", "function": {
        "name": "read_table",
        "description": "用视觉模型读取图片里的评分表（含Tot.列），返回表格、读到的总分，以及警告。"
                       "警告包括行长度不一致、以及校验和不符（某行分数之和不等于Tot.，说明该行有数字读错）。有警告时请带 hint 重读。",
        "parameters": {"type": "object", "properties": {
            "image_path": {"type": "string"},
            "hint": {"type": "string", "description": "可选，告诉读图模型哪些行读错了、需要逐格核对"}},
            "required": ["image_path"]}}},
    COMPUTE_SCHEMA,
]
SYSTEM_V2 = ("你是处理评分表图片的助手。先用 read_table 读图。如果返回有 warnings（行长度不一致或校验和不符），"
             "必须带 hint 重读（说明哪些行有问题）；最多重读3次。读表没有警告后，用 compute_totals 计算，不要心算；"
             "如果结果超过 max_possible_per_song 或明显不合理，回头检查。最后一行以“最终答案：”开头，只写一个数字。")

def run_scripted_v2(b, image_path, max_retries=2):
    hint, reads, out = "", 0, {}
    for _ in range(max_retries + 1):
        out = read_table_v2(image_path, hint); reads += 1
        if "table" not in out:
            hint = "上一次输出不是合法JSON，请严格只输出JSON。"; continue
        if not out["warnings"]:
            break
        hint = f"上一次读取有问题：{'; '.join(out['warnings'])}。请逐格重新核对每个数字。"
    if "table" not in out:
        return None, out, reads
    return solve(b, out["table"]), out, reads

exp3 = []
for lvl in ["mild", "hard"]:
    for i, b in enumerate(bench):
        dst = f"/content/eval/g{i+1}_{lvl}.jpg"
        sc_ans, sc_out, sc_reads = run_scripted_v2(b, dst)
        res = run_agent(b["question"], dst, TOOLS_V2, TOOL_FUNCS_V2, SYSTEM_V2, max_steps=10)
        reads, last = reads_of(res["steps"])
        exp3.append({"level": lvl, "i": i, "truth": b["answer"],
                     "scripted": sc_ans, "scripted_ok": is_ok(sc_ans, b["answer"]),
                     "scripted_exact": ("table" in sc_out) and sc_out["table"] == b["true_table"], "scripted_reads": sc_reads,
                     "agent": res["answer"], "agent_ok": is_ok(res["answer"], b["answer"]),
                     "agent_exact": last == b["true_table"], "agent_reads": len(reads),
                     "agent_hints": [s["args"].get("hint", "") for s in reads],
                     "agent_warnings": [s["result"].get("warnings") for s in reads],
                     "cost": res["cost"], "final": res["final"]})
        r = exp3[-1]
        print(f"[{lvl}] Q{i+1}: truth={r['truth']} | scripted={r['scripted']} {'OK' if r['scripted_ok'] else 'X'} "
              f"(exact={r['scripted_exact']}, reads={r['scripted_reads']}) | agent={r['agent']} "
              f"{'OK' if r['agent_ok'] else 'X'} (exact={r['agent_exact']}, reads={r['agent_reads']})")
print()
for lvl in ["mild", "hard"]:
    r1 = [r for r in exp2 if r["level"] == lvl]
    r3 = [r for r in exp3 if r["level"] == lvl]
    print(f"{lvl}: fixed_v1 {sum(r['fixed_ok'] for r in r1)}/8 | agent_v1 {sum(r['agent_ok'] for r in r1)}/8 | "
          f"scripted_v2 {sum(r['scripted_ok'] for r in r3)}/8 (table exact {sum(r['scripted_exact'] for r in r3)}/8) | "
          f"agent_v2 {sum(r['agent_ok'] for r in r3)}/8 (table exact {sum(r['agent_exact'] for r in r3)}/8)")
print(f"agent_v2 cost ~ ${sum(r['cost'] for r in exp3):.4f}")
json.dump(exp3, open("/content/exp3_checksum.json", "w"), ensure_ascii=False, indent=2)

## 7. Experiment 4: diagnosing the reader (Qwen only, no API cost)

How often is the **first** read of the table exactly right, with or without the `Tot.` column in the prompt, on the original image vs a 2x upscaled copy?

In [ ]:
diag = []
for lvl in ["mild", "hard"]:
    for i, b in enumerate(bench):
        src = f"/content/eval/g{i+1}_{lvl}.jpg"
        up = f"/content/eval/g{i+1}_{lvl}_x2.jpg"
        upscale(src, up)
        row = {"level": lvl, "i": i}
        for name, path in [("orig", src), ("x2", up)]:
            o1, o2 = read_table(path), read_table_v2(path)
            row[f"v1_{name}"] = ("table" in o1 and o1["table"] == b["true_table"])
            row[f"v2_{name}"] = ("table" in o2 and o2["table"] == b["true_table"])
        diag.append(row)
for lvl in ["mild", "hard"]:
    rs = [r for r in diag if r["level"] == lvl]
    print(f"{lvl}: first read exactly right (of 8) | without Tot.: orig {sum(r['v1_orig'] for r in rs)}, x2 {sum(r['v1_x2'] for r in rs)} | "
          f"with Tot.: orig {sum(r['v2_orig'] for r in rs)}, x2 {sum(r['v2_x2'] for r in rs)}")
json.dump(diag, open("/content/exp4_diagnostic.json", "w"), indent=2)

## 8. Experiment 5: multi-view consistency on a fresh test set (Qwen only, no API cost)

The methods were fixed **before** looking at this set (new seed, 12 new tables x 2 difficulty levels), to avoid tuning on the
tables used above.
- **A**: read the original image once.
- **B**: read a 2x upscaled copy once.
- **C**: read original, 2x and 3x; use a table if at least two views agree, otherwise fall back to the 2x read.

Also: does agreement between views predict a correct read?

In [ ]:
fresh = make_bench(123, 12, "f")

def table_of(path):
    return read_table(path).get("table")

def majority(tabs):
    keyed = [json.dumps(t, sort_keys=True) if t else None for t in tabs]
    for key in keyed:
        if key and keyed.count(key) >= 2:
            return json.loads(key), True
    return (tabs[1] or tabs[0]), False

exp5 = []
for lvl in ["mild", "hard"]:
    for i, b in enumerate(fresh):
        src = b["image"].replace(".png", f"_{lvl}.jpg")
        degrade(b["image"], src, lvl)
        tabs = []
        for k in (1, 2, 3):
            if k == 1:
                tabs.append(table_of(src))
            else:
                up = src.replace(".jpg", f"_x{k}.jpg")
                upscale(src, up, k)
                tabs.append(table_of(up))
        c, agreed = majority(tabs)
        row = {"level": lvl, "i": i, "truth": b["answer"], "agreed": agreed}
        for name, t in [("A", tabs[0]), ("B", tabs[1]), ("C", c)]:
            ans = solve(b, t) if t else None
            row[name + "_exact"] = (t == b["true_table"])
            row[name + "_ok"] = is_ok(ans, b["answer"])
        exp5.append(row)

for lvl in ["mild", "hard"]:
    rs = [r for r in exp5 if r["level"] == lvl]
    print(f"{lvl} (n={len(rs)}) | table exact A {sum(r['A_exact'] for r in rs)}, B {sum(r['B_exact'] for r in rs)}, "
          f"C {sum(r['C_exact'] for r in rs)} | answer correct A {sum(r['A_ok'] for r in rs)}, "
          f"B {sum(r['B_ok'] for r in rs)}, C {sum(r['C_ok'] for r in rs)} | two views agree on {sum(r['agreed'] for r in rs)}")
print()
for lvl in ["mild", "hard"]:
    for flag in [True, False]:
        rs = [r for r in exp5 if r["level"] == lvl and r["agreed"] == flag]
        if rs:
            print(lvl, "two views agree" if flag else "no agreement", len(rs), "images -> C table exact", sum(r["C_exact"] for r in rs))
json.dump(exp5, open("/content/exp5_fresh.json", "w"), indent=2)

## Notes
- The DeepSeek calls are sampled, so re-running the agent experiments can change individual outcomes. Qwen is greedy and deterministic.
- Download the `/content/exp*.json` files if you want to keep the raw per-question results.